# Imports

In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
BF16 supported: True


In [2]:
!pip install -q -U transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 124.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 20.9 MB/s eta 0:00:00


In [3]:
!pip install -q pandas==2.2.2

In [4]:
!pip install -q psutil

In [5]:
import pandas as pd
import transformers
import bitsandbytes as bnb

print("Pandas:", pd.__version__)
print("Transformers:", transformers.__version__)
print("BitsAndBytes:", bnb.__version__)

Pandas: 2.2.2
Transformers: 5.14.1
BitsAndBytes: 0.50.0


In [6]:
import gc
import json
import time
from pathlib import Path
from statistics import mean

import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
import os
import psutil

# Configuration

In [7]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

MAX_NEW_TOKENS = 160

FULL_PRECISION_DTYPE = torch.bfloat16

RESULTS_DIR = Path("results")
OUTPUTS_DIR = RESULTS_DIR / "outputs"

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# prompts

In [8]:
PROMPTS = [
    {
        "id": "explanation",
        "prompt": "Explain retrieval-augmented generation to a beginner in less than 120 words."
    },
    {
        "id": "reasoning",
        "prompt": (
            "A support team receives 600 tickets daily. "
            "An AI resolves 65% automatically. "
            "40% of the remaining tickets require escalation. "
            "How many tickets are escalated?"
        )
    },
    {
        "id": "coding",
        "prompt": (
            "Write a Python function that removes duplicate dictionaries "
            "based on the key 'id' while preserving order."
        )
    },
    {
        "id": "summarization",
        "prompt": (
            "Summarize the benefits and drawbacks of LLM quantization in three bullet points."
        )
    },
    {
        "id": "instruction_following",
        "prompt": (
            "Give exactly three advantages of Docker for AI deployment."
        )
    }
]

# Tokenizer

In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

# Utility Function

In [10]:
def gb(x):
    return round(x / 1024**3, 3)


def reset_cuda():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

process = psutil.Process(os.getpid())

def memory():
    torch.cuda.synchronize()

    ram = process.memory_info().rss

    return {
        "ram_gb": gb(ram),
        "allocated_gb": gb(torch.cuda.memory_allocated()),
        "reserved_gb": gb(torch.cuda.memory_reserved()),
        "peak_allocated_gb": gb(torch.cuda.max_memory_allocated()),
        "peak_reserved_gb": gb(torch.cuda.max_memory_reserved()),
    }

# Chat Template

In [11]:
def build_inputs(prompt):

    messages = [
        {
            "role": "system",
            "content": "You are a helpful AI assistant."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    return tokenizer(
        formatted,
        return_tensors="pt"
    )

# Benchmark Function

In [12]:
def benchmark_model(
    model,
    model_name,
    prompts,
    max_new_tokens=MAX_NEW_TOKENS,
):
    model.eval()

    all_results = []

    # -------------------------------
    # Warm-up
    # -------------------------------

    warmup = build_inputs("Say hello in one short sentence.")

    warmup = {
        k: v.to(model.device)
        for k, v in warmup.items()
    }

    with torch.inference_mode():
        model.generate(
            **warmup,
            max_new_tokens=16,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    torch.cuda.synchronize()

    load_memory = memory()

    # -------------------------------
    # Benchmark prompts
    # -------------------------------

    for item in prompts:

        reset_cuda()

        encoded = build_inputs(item["prompt"])

        encoded = {
            k: v.to(model.device)
            for k, v in encoded.items()
        }

        input_tokens = encoded["input_ids"].shape[-1]

        start = time.perf_counter()

        with torch.inference_mode():

            outputs = model.generate(
                **encoded,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        torch.cuda.synchronize()

        elapsed = time.perf_counter() - start

        generated_ids = outputs[0][input_tokens:]

        output = tokenizer.decode(
            generated_ids,
            skip_special_tokens=True,
        )

        generated_tokens = len(generated_ids)

        tps = (
            generated_tokens / elapsed
            if elapsed > 0
            else 0
        )

        all_results.append(
            {
                "id": item["id"],
                "prompt": item["prompt"],
                "input_tokens": input_tokens,
                "output_tokens": generated_tokens,
                "output": output.strip(),
                "latency_seconds": round(elapsed, 4),
                "tokens_per_second": round(tps, 3),
                "memory": memory(),
            }
        )

    return {
        "model": model_name,
        "average_latency": round(
            mean(
                x["latency_seconds"]
                for x in all_results
            ),
            4,
        ),
        "average_tokens_per_second": round(
            mean(
                x["tokens_per_second"]
                for x in all_results
            ),
            3,
        ),
        "load_memory": load_memory,
        "results": all_results,
    }

# Loading Full Precision

In [13]:
reset_cuda()

full_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=FULL_PRECISION_DTYPE,
    device_map="auto",
    low_cpu_mem_usage=True,
)

full_results = benchmark_model(
    full_model,
    "BF16",
    PROMPTS,
)

print(json.dumps(
    {
        "Average Latency": full_results["average_latency"],
        "Average TPS": full_results["average_tokens_per_second"],
        "Memory": full_results["load_memory"],
    },
    indent=2,
))

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

{
  "Average Latency": 5.0919,
  "Average TPS": 27.072,
  "Memory": {
    "ram_gb": 1.923,
    "allocated_gb": 2.884,
    "reserved_gb": 2.934,
    "peak_allocated_gb": 2.887,
    "peak_reserved_gb": 2.934
  }
}


# Clean GPU

In [14]:
del full_model

gc.collect()

torch.cuda.empty_cache()

torch.cuda.synchronize()

print(memory())

{'ram_gb': 1.924, 'allocated_gb': 0.009, 'reserved_gb': 2.877, 'peak_allocated_gb': 2.893, 'peak_reserved_gb': 2.936}


# Loading NF4

In [15]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

reset_cuda()

quant_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)

quant_results = benchmark_model(
    quant_model,
    "4-bit NF4",
    PROMPTS,
)

print(json.dumps(
    {
        "Average Latency": quant_results["average_latency"],
        "Average TPS": quant_results["average_tokens_per_second"],
        "Memory": quant_results["load_memory"],
    },
    indent=2,
))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

{
  "Average Latency": 7.537,
  "Average TPS": 18.752,
  "Memory": {
    "ram_gb": 1.939,
    "allocated_gb": 1.084,
    "reserved_gb": 2.912,
    "peak_allocated_gb": 1.113,
    "peak_reserved_gb": 2.912
  }
}


# Benchmark Result

In [16]:
benchmark_results = {
    "model_id": MODEL_ID,
    "hardware": {
        "gpu": torch.cuda.get_device_name(0),
        "cuda_version": torch.version.cuda,
        "pytorch_version": torch.__version__,
    },
    "configurations": {
        "bf16": full_results,
        "4bit_nf4": quant_results,
    },
}

results_path = RESULTS_DIR / "benchmark_results.json"

with open(results_path, "w", encoding="utf-8") as file:
    json.dump(
        benchmark_results,
        file,
        indent=2,
        ensure_ascii=False,
    )

print(f"Saved results to: {results_path}")

Saved results to: results/benchmark_results.json


In [17]:
import pandas as pd

comparison_df = pd.DataFrame([
    {
        "Configuration": "BF16",
        "Average Latency (s)": full_results["average_latency"],
        "Throughput (tokens/s)": full_results["average_tokens_per_second"],
        "RAM (GB)": full_results["load_memory"]["ram_gb"],
        "Allocated VRAM (GB)": full_results["load_memory"]["allocated_gb"],
        "Peak VRAM (GB)": full_results["load_memory"]["peak_allocated_gb"],
    },
    {
        "Configuration": "4-bit NF4",
        "Average Latency (s)": quant_results["average_latency"],
        "Throughput (tokens/s)": quant_results["average_tokens_per_second"],
        "RAM (GB)": quant_results["load_memory"]["ram_gb"],
        "Allocated VRAM (GB)": quant_results["load_memory"]["allocated_gb"],
        "Peak VRAM (GB)": quant_results["load_memory"]["peak_allocated_gb"],
    },
])

comparison_df

,Configuration,Average Latency (s),Throughput (tokens/s),RAM (GB),Allocated VRAM (GB),Peak VRAM (GB)
0,BF16,5.0919,27.072,1.923,2.884,2.887
1,4-bit NF4,7.5370,18.752,1.939,1.084,1.113


In [18]:
bf16_output_path = OUTPUTS_DIR / "full_precision.md"

with open(bf16_output_path, "w", encoding="utf-8") as f:

    f.write("# BF16 Outputs\n\n")

    for item in full_results["results"]:

        f.write(f"## {item['id']}\n\n")

        f.write("### Prompt\n")

        f.write(item["prompt"] + "\n\n")

        f.write("### Response\n")

        f.write(item["output"] + "\n\n")

        f.write("---\n\n")

print(bf16_output_path)

results/outputs/full_precision.md


In [19]:
nf4_output_path = OUTPUTS_DIR / "quantized_4bit.md"

with open(nf4_output_path, "w", encoding="utf-8") as f:

    f.write("# 4-bit NF4 Outputs\n\n")

    for item in quant_results["results"]:

        f.write(f"## {item['id']}\n\n")

        f.write("### Prompt\n")

        f.write(item["prompt"] + "\n\n")

        f.write("### Response\n")

        f.write(item["output"] + "\n\n")

        f.write("---\n\n")

print(nf4_output_path)

results/outputs/quantized_4bit.md


In [20]:
comparison = f"""# Quantization Benchmark Comparison

## Model

{MODEL_ID}

## Hardware

- GPU: {torch.cuda.get_device_name(0)}
- CUDA: {torch.version.cuda}
- PyTorch: {torch.__version__}

---

## Benchmark Results

| Metric | BF16 | 4-bit NF4 |
|--------|------|-----------|
| Average Latency (s) | {full_results['average_latency']} | {quant_results['average_latency']} |
| Throughput (tokens/sec) | {full_results['average_tokens_per_second']} | {quant_results['average_tokens_per_second']} |
| RAM (GB) | {full_results['load_memory']['ram_gb']} | {quant_results['load_memory']['ram_gb']} |
| Allocated VRAM (GB) | {full_results['load_memory']['allocated_gb']} | {quant_results['load_memory']['allocated_gb']} |
| Peak VRAM (GB) | {full_results['load_memory']['peak_allocated_gb']} | {quant_results['load_memory']['peak_allocated_gb']} |

---

## Analysis

The NF4 quantized model reduced GPU memory consumption significantly,
decreasing allocated VRAM from approximately
{full_results['load_memory']['allocated_gb']} GB
to
{quant_results['load_memory']['allocated_gb']} GB.

However, for this benchmark on a Tesla T4 GPU using
Qwen2.5-1.5B-Instruct,
the BF16 model achieved lower latency and higher throughput.

This demonstrates the common trade-off of quantization:

- Lower memory footprint
- Ability to run on smaller GPUs
- Possible reduction in inference speed depending on hardware and model size

For this workload, BF16 provides the best inference performance,
while NF4 offers substantially better memory efficiency.
"""

comparison_path = RESULTS_DIR / "comparison.md"

with open(comparison_path, "w", encoding="utf-8") as f:
    f.write(comparison)

print(comparison_path)

results/comparison.md


In [21]:
import shutil

shutil.make_archive(
    "results",
    "zip",
    "results"
)

'/content/results.zip'